# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imnxr/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Data Loading and Signal Exploration

In [1]:
import os

for item in os.listdir("../../data"):
    print(item)

raw


In [2]:
import os

for root, dirs, files in os.walk("../../data/raw"):
    print("Folder:", root)
    for file in files[:10]:
        print("  ", file)

Folder: ../../data/raw
   content_refresh_anonymized.csv


In [3]:
import pandas as pd

path = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [5]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
content_id,30000,30000,content_304f48230142,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
client_id,30000,32,client_19581e27de,7008,NaN,NaN,NaN,NaN,NaN,NaN,NaN
search_volume,27532.0,NaN,NaN,NaN,158.882391,1518.270825,0.0,0.0,10.0,20.0,74000.0
competition,27532.0,NaN,NaN,NaN,0.146954,0.285241,0.0,0.0,0.0,0.13,1.0
competition_level,27390,3,LOW,22896,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpc,27532.0,NaN,NaN,NaN,0.485342,2.10156,0.0,0.0,0.0,0.0,100.36
content_type,30000,3,keyword article,27207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
main_intent,27626,4,informational,17235,NaN,NaN,NaN,NaN,NaN,NaN,NaN
word_count,22301.0,NaN,NaN,NaN,3107.760325,1452.382598,8.0,2413.0,2877.0,3666.0,9546.0
char_count,22301.0,NaN,NaN,NaN,20665.277835,10115.344042,40.0,15644.0,19116.0,24011.0,111158.0


In [6]:
df["trend_direction"].value_counts()

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

In [7]:
df["engagement_rate"].describe()

count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The baseline action score uses two signals:

1. Trend direction:
   - Downward trends indicate declining content performance and increase refresh priority.
   - Upward or stable trends reduce the need for immediate action.

2. Engagement rate:
   - Low engagement indicates users are not interacting with the content.
   - Higher engagement indicates content is still valuable.

Reason codes:

- HIGH_REFRESH_PRIORITY:
  Content has declining trend and low engagement.

- LOW_ENGAGEMENT:
  Content has poor engagement but no strong trend decline.

- STABLE_CONTENT:
  Content is performing consistently and requires no urgent action.

- GROWING_CONTENT:
  Content has positive growth signals.

In [8]:
def baseline_rule(row):

    if row["trend_direction"] == "down" and row["engagement_rate"] < 1:
        return pd.Series({
            "score": 0.9,
            "reason_code": "HIGH_REFRESH_PRIORITY",
            "action_label": "REFRESH"
        })

    elif row["engagement_rate"] < 1:
        return pd.Series({
            "score": 0.7,
            "reason_code": "LOW_ENGAGEMENT",
            "action_label": "REVIEW"
        })

    elif row["trend_direction"] in ["up", "new"]:
        return pd.Series({
            "score": 0.2,
            "reason_code": "GROWING_CONTENT",
            "action_label": "KEEP"
        })

    else:
        return pd.Series({
            "score": 0.5,
            "reason_code": "STABLE_CONTENT",
            "action_label": "MONITOR"
        })

In [9]:
test = df.head(5).apply(baseline_rule, axis=1)

test

,score,reason_code,action_label
0,0.5,STABLE_CONTENT,MONITOR
1,0.9,HIGH_REFRESH_PRIORITY,REFRESH
2,0.9,HIGH_REFRESH_PRIORITY,REFRESH
3,0.5,STABLE_CONTENT,MONITOR
4,0.9,HIGH_REFRESH_PRIORITY,REFRESH


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:

scores = df.apply(baseline_rule, axis=1)

# Combine original data with predictions
baseline_df = pd.concat([df, scores], axis=1)

# Rank by score (highest priority first)
baseline_df = baseline_df.sort_values(
    by="score",
    ascending=False
)

baseline_df.head()


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action_label
28,content_19ad8f9bac29,client_3fdba35f04,480.0,0.58,MEDIUM,1.97,keyword article,informational,1372.0,9052.0,...,0.0,58.33,0.0,moderate,deep,down,-60.0,0.9,HIGH_REFRESH_PRIORITY,REFRESH
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.0,10.00,0.0,good,page_3_5,down,-57.7,0.9,HIGH_REFRESH_PRIORITY,REFRESH
29983,content_6880eb215048,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,transactional,NaN,NaN,...,0.0,0.00,0.0,moderate,page_1,down,-21.8,0.9,HIGH_REFRESH_PRIORITY,REFRESH
27,content_7ea135180dd9,client_4ec9599fc2,10.0,0.32,LOW,10.57,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,moderate,page_1,down,-64.7,0.9,HIGH_REFRESH_PRIORITY,REFRESH
25,content_033ae3e7aecf,client_f369cb89fc,70.0,0.81,HIGH,1.12,keyword article,commercial,2777.0,16215.0,...,0.0,0.00,50.0,low,page_1,down,-35.7,0.9,HIGH_REFRESH_PRIORITY,REFRESH


In [11]:
import os

# Create output folder if it doesn't exist
os.makedirs("../../work/outputs", exist_ok=True)

output_path = "../../work/outputs/baseline_action_score.csv"

baseline_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: ../../work/outputs/baseline_action_score.csv


In [16]:
import os

csv_path = "../../work/outputs/baseline_action_score.csv"

print("CSV exists:", os.path.exists(csv_path))

if os.path.exists(csv_path):
    print("Rows:", pd.read_csv(csv_path).shape[0])

CSV exists: True
Rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
top20 = baseline_df.head(20)

top20[
    [
        "content_id",
        "score",
        "reason_code",
        "action_label",
        "trend_direction",
        "engagement_rate",
        "trend_pct"
    ]
]


,content_id,score,reason_code,action_label,trend_direction,engagement_rate,trend_pct
28,content_19ad8f9bac29,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-60.0
1,content_a1fb4e703a9e,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-57.7
29983,content_6880eb215048,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-21.8
27,content_7ea135180dd9,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-64.7
25,content_033ae3e7aecf,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-35.7
29977,content_c87291853cab,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-33.3
29988,content_9bb9a0584cae,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-100.0
29987,content_7b36d6ffe9bd,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-61.5
14,content_91067a14431a,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-64.9
9,content_c27558df2b0c,0.9,HIGH_REFRESH_PRIORITY,REFRESH,down,0.0,-29.2


## Top-20 Review Notes

The baseline ranking prioritizes pages with declining performance signals.

For each top-ranked item:

**Action:**
Refresh content.

**Reason code:**
HIGH_REFRESH_PRIORITY.

**Confidence:**
High confidence because two independent signals agree:
- The content trend is declining (`trend_direction = down`)
- User engagement is very low (`engagement_rate = 0`)

**What would make this wrong:**
- The traffic decline may be caused by seasonality.
- A recent external event may temporarily reduce traffic.
- The page may have intentionally reduced visibility.
- Engagement tracking may be incomplete.

In [14]:
review = top20[
    [
        "content_id",
        "action_label",
        "reason_code"
    ]
].copy()

review["confidence_note"] = (
    "High: declining trend and low engagement agree"
)

review["what_would_make_it_wrong"] = (
    "Seasonality, external events, or incomplete tracking"
)

review

,content_id,action_label,reason_code,confidence_note,what_would_make_it_wrong
28,content_19ad8f9bac29,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
1,content_a1fb4e703a9e,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
29983,content_6880eb215048,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
27,content_7ea135180dd9,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
25,content_033ae3e7aecf,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
29977,content_c87291853cab,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
29988,content_9bb9a0584cae,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
29987,content_7b36d6ffe9bd,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
14,content_91067a14431a,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."
9,content_c27558df2b0c,REFRESH,HIGH_REFRESH_PRIORITY,High: declining trend and low engagement agree,"Seasonality, external events, or incomplete tr..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



### Weak picks

Some refresh recommendations may be incorrect because:

- A declining trend may be caused by seasonal changes rather than content quality.
- Low engagement may come from low search demand instead of poor content.
- Some pages may intentionally target a small audience.
- A page with low engagement may still be valuable if it serves a narrow audience.

### Leakage check

The baseline rule only uses currently available signals:

- trend_direction
- engagement_rate

It does not use future performance, future clicks, future rankings, or post-action outcomes.

No future-window or label-derived features were used.

In [15]:
# Check features used by the rule

used_features = [
    "trend_direction",
    "engagement_rate"
]

print("Features used:")
for feature in used_features:
    print("-", feature)

print("\nNo future labels or outcome columns used.")

Features used:
- trend_direction
- engagement_rate

No future labels or outcome columns used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.